In [ ]:
# ============================================================
# kaggle/notebooks/phase0a_qaoa_rerun.ipynb
# Single-cell notebook — paste into a NEW cell AFTER the main sweep
# ============================================================
import sys, subprocess, os, shutil, re, importlib.util, json, glob

REPO_DIR = "opensim-ai"
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/kaggle")

# -----------------------------------------------------------
# 1. Pull latest code (QAOA fix must be live)
# -----------------------------------------------------------
subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
print("[OK] Repo up to date")

# -----------------------------------------------------------
# 2. Reload modules (Kaggle runner + assembler)
# -----------------------------------------------------------
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

runner_mod    = load_module("kaggle_runner",    f"{REPO_DIR}/kaggle/runner.py")
assembler_mod = load_module("kaggle_assembler", f"{REPO_DIR}/kaggle/dataset_assembler.py")
client_mod    = load_module("kaggle_client",    f"{REPO_DIR}/kaggle/api_client.py")

KaggleRunner     = runner_mod.KaggleRunner
DatasetAssembler = assembler_mod.DatasetAssembler
KaggleAPIClient  = client_mod.KaggleAPIClient

CKPT_DIR  = "/kaggle/working/checkpoints"
RAW_DIR   = "/kaggle/working/benchmark_outputs/raw"
CKPT_FILE = f"{CKPT_DIR}/sweep_checkpoint.json"
COMPL_FILE= f"{CKPT_DIR}/completed_combinations.txt"

# Load Kaggle username from secrets
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        _v = _s.get_secret(_k)
        if _v:
            os.environ[_k] = _v
except Exception:
    pass
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "harshalekkala1")
KAGGLE_DATASET  = f"{KAGGLE_USERNAME}/openqsim-benchmarks"
print(f"[OK] Target dataset: {KAGGLE_DATASET}")

# Sweep layout (sweep_config_0a.yaml):
#   ghz   -> indices   0-209  (210 combos)
#   qft   -> indices 210-419
#   random-> indices 420-629
#   qaoa  -> indices 630-839  <-- re-run these
QAOA_START_IDX = 630

# -----------------------------------------------------------
# 3. Delete all failed QAOA JSONs
# -----------------------------------------------------------
print("[1/4] Deleting failed QAOA records...")
deleted = 0
for jf in glob.glob(f"{RAW_DIR}/qaoa_*.json"):
    try:
        with open(jf) as f:
            rec = json.load(f)
        if not rec.get("success", True):
            os.remove(jf)
            deleted += 1
    except Exception:
        pass
print(f"      Deleted {deleted} failed QAOA files.")

# -----------------------------------------------------------
# 4. Remove QAOA combo_keys from checkpoint
# -----------------------------------------------------------
print("[2/4] Cleaning checkpoint...")
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        ckpt = json.load(f)
else:
    ckpt = {"last_completed_index": -1, "completed_count": 0,
            "oom_count": 0, "error_count": 0}

if os.path.exists(COMPL_FILE):
    with open(COMPL_FILE) as f:
        lines = f.read().splitlines()
    new_lines = [l for l in lines if not l.startswith("qaoa_")]
    with open(COMPL_FILE, "w") as f:
        f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
    removed = len(lines) - len(new_lines)
    print(f"      Removed {removed} QAOA entries from completed_combinations.txt")

# Roll back to just before QAOA range so runner starts exactly at index 630.
if ckpt.get("last_completed_index", 0) >= QAOA_START_IDX:
    ckpt["last_completed_index"] = QAOA_START_IDX - 1  # 629
ckpt["completed_count"] = max(0, ckpt.get("completed_count", 0) - deleted)
ckpt["error_count"]     = max(0, ckpt.get("error_count",     0) - deleted)
with open(CKPT_FILE, "w") as f:
    json.dump(ckpt, f, indent=2)
print(f"      Checkpoint index set to {ckpt['last_completed_index']}, "
      f"completed_count={ckpt['completed_count']}")

# -----------------------------------------------------------
# 5. Re-run QAOA combos (210 total)
# -----------------------------------------------------------
print("[3/4] Re-running QAOA with fixed generator...")
runner = KaggleRunner(
    sweep_config_path  = f"{REPO_DIR}/benchmark/sweep_config_0a.yaml",
    output_dir         = "/kaggle/working/benchmark_outputs",
    checkpoint_interval= 10,
    artifact_interval  = 50,
    kaggle_dataset     = KAGGLE_DATASET,
    use_advisor        = False,
)
result = runner.run()
print(f"      QAOA re-run done: {result.completed_count} total records, "
      f"{result.error_count} errors")

# -----------------------------------------------------------
# 6. Re-assemble dataset & push
# -----------------------------------------------------------
print("[4/4] Re-assembling final dataset...")
assembler = DatasetAssembler(
    raw_dir    = RAW_DIR,
    dataset_dir= "/kaggle/working/benchmark_outputs/datasets/openqsim_v0.1-small",
)
manifest = assembler.assemble()

client = KaggleAPIClient(dataset=KAGGLE_DATASET)
client.push_benchmark_outputs(
    "/kaggle/working/benchmark_outputs",
    message=f"Phase 0A final — {manifest.records} records"
)
print(f"Final dataset: {manifest.records} records, {manifest.successful_runs} successful")
